In [1]:
%pip install beautifulsoup4 scikit-learn selenium

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Alen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit
from sklearn.feature_extraction.text import CountVectorizer
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
from threading import Thread

url = "https://www.24ur.com/novice/tujina/zadeti-ruski-tanker-brez-posadke-pluje-po-sredozemlju.html"

driver = webdriver.Chrome()
driver.get(url)
try:
    proad = driver.find_element(By.ID, "proad")
    ActionChains(driver).scroll_to_element(proad).perform()
    wait = WebDriverWait(driver, timeout=3)
    wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "proad-stat")))
except:
    print("proad-stat doesn't exist, skipping")
source = driver.page_source
driver.close()


In [4]:
def title_relevance_score(url_title, keywords):
    title_vector = CountVectorizer(url_title)
    keywords_vector = CountVectorizer(keywords)
    print(title_vector)
    print(keywords_vector)

In [29]:
def get_topic_from_url(url):
    url_parts = get_url_parts(url)
    try:
        topic = url_parts[2]
    except:
        return None
    return topic

def get_url_parts(url, delim="/"):
    return urlsplit(url).path.split(delim)

def extract_urls(html, url):
    bs = BeautifulSoup(html)

    metadata_dict = dict()

    # extract article 
    

    #print(bs.prettify())
    links = bs.find_all("a", href=True)
    print(f"Num of all links: {len(links)}")
    url_parts = get_url_parts(url)
    for l in links:
        key = urljoin(url, l["href"])
        key_parts = get_url_parts(key)
        #print(url_parts)
        if len(key_parts) < 2:
            #print("URL parts too short")
            continue
        if key_parts[1] == "s":
            continue
        if "24ur.com" not in key:
            continue
        metadata_dict[key] = dict({
            "source": url,
            "source_title": url_parts[-1] if ".html" in url else None,
            "section": key_parts[1] if len(key_parts) > 1 else None,
            "topic": key_parts[2] if len(key_parts) > 2 else None,
            "container_id": None
        })

        if key_parts[1] == "kljucna-beseda":
            metadata_dict[key]["container_id"] = "kljucna-beseda"

        if key_parts[1] == "spored":
            metadata_dict[key]["container_id"] = "spored"
        # if key_parts[1] == "vreme":
        #     print("Vreme")

        parent_3 = l.parent.parent.parent
        parent_2 = l.parent.parent
        parent_1 = l.parent

        if parent_3.has_attr("class"):
            if "menu__items" in parent_3.attrs["class"]:
                metadata_dict[key]["container_id"] = "menu"
            elif "submenu" in parent_3.attrs["class"]:
                metadata_dict[key]["container_id"] = "submenu"
        # print(parent_2.attrs)
        if parent_2.has_attr("id"):
            if parent_2.attrs["id"] == "footer-bottom":
                metadata_dict[key]["container_id"] = "footer"
            


    # related articles
    related_articles = bs.find(id="related-articles")
    if related_articles != None:
        related_links = related_articles.find_all("a", href=True)
        print("Related articles:")
        for l in related_links:
            key = urljoin(url, l["href"])
            if key not in metadata_dict:
                continue
            metadata_dict[key]["container_id"] = "related-articles"
            print(key)

    # Read more links
    read_mores = bs.find_all("a", class_="read-more")
    print("Read mores:")
    for l in read_mores:
        key = urljoin(url, l["href"])
        if key not in metadata_dict:
            continue
        metadata_dict[key]["container_id"] = "read-more"
        print(key)

    # Proad recommends
    print("Proad links: ")
    proads_div = bs.find(id="proad")
    if proads_div != None:
        proads_links = proads_div.find_all("a", href=True)
        for l in proads_links:
            if l["href"] == "/vsebine/oglasevanje":
                continue
            key = urljoin(url, l["href"])
            if key not in metadata_dict:
                continue
            metadata_dict[key]["container_id"] = "proad"
            print(key)
    return metadata_dict
extract_urls(source, url)

Num of all links: 271
Read mores:
https://www.24ur.com/novice/tujina/rusija-ukrajina-z-brezpilotniki-potopila-tanker-v-sredozemlju.html
Proad links: 
https://go-ctr-tracker.pub.24ur.si/rec/JS_1/c/0/6c09542e-0a27-4ddd-a3ac-3951a827d73d/1.gif?articleId=4435549&at=1773515098&mobile=1&redir=https%3A%2F%2Fwww.24ur.com%2Fnovice%2Ftujina%2Fmalta-prepovedala-vplutje-ladji-kathrin.html%3Futm_source%3DProAd%26utm_medium%3D24ur%26utm_content%3DProAd_24ur__%26utm_campaign%3DProAd&source=vector&sig=c739fa2cadf7547fd16f4fc4b98a638c37249e423e19968ce12d16287d2d9208
https://go-ctr-tracker.pub.24ur.si/rec/JS_1/c/2/6c09542e-0a27-4ddd-a3ac-3951a827d73d/1.gif?articleId=4435342&at=1773515098&mobile=1&redir=https%3A%2F%2Fwww.24ur.com%2Fnovice%2Fslovenija%2Fkathrin-ne-bo-v-koper-pluje-proti-malti.html%3Futm_source%3DProAd%26utm_medium%3D24ur%26utm_content%3DProAd_24ur__%26utm_campaign%3DProAd&source=vector&sig=8965787f7aef1212ae4835c83223715ba2723f4fe03d0a2a7331575820b30e91
https://go-ctr-tracker.pub.24ur.si/

{'https://www.24ur.com/': {'source': 'https://www.24ur.com/novice/tujina/zadeti-ruski-tanker-brez-posadke-pluje-po-sredozemlju.html',
  'source_title': 'zadeti-ruski-tanker-brez-posadke-pluje-po-sredozemlju.html',
  'section': '',
  'topic': None,
  'container_id': None},
 'https://www.24ur.com/novice/slovenija-odloca': {'source': 'https://www.24ur.com/novice/tujina/zadeti-ruski-tanker-brez-posadke-pluje-po-sredozemlju.html',
  'source_title': 'zadeti-ruski-tanker-brez-posadke-pluje-po-sredozemlju.html',
  'section': 'novice',
  'topic': 'slovenija-odloca',
  'container_id': 'menu'},
 'https://www.24ur.com/novice/slovenija': {'source': 'https://www.24ur.com/novice/tujina/zadeti-ruski-tanker-brez-posadke-pluje-po-sredozemlju.html',
  'source_title': 'zadeti-ruski-tanker-brez-posadke-pluje-po-sredozemlju.html',
  'section': 'novice',
  'topic': 'slovenija',
  'container_id': 'submenu'},
 'https://www.24ur.com/novice/tujina': {'source': 'https://www.24ur.com/novice/tujina/zadeti-ruski-tan

Metdata o linkih
- title originalnega page-a
- title linka
- topic/tag source page-a
- container/div id
- article keywords